# SecureFace-RX — 영상 배치 익명화/복원 (코랩 GPU)

CPU 대비 5~10배 빠르게 영상 전체를 익명화하고 복원합니다.

**실행 전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **하드웨어 가속기: T4 GPU** 선택
2. 아래 셀을 위에서부터 차례로 실행 (Shift+Enter)

**업로드할 파일** (4-1, 4-2 셀에서 따로따로)
- `demo.mp4` — 처리할 영상
- `hybridAll_inv3_...iter15000.pth` — INN 체크포인트

## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
print('torch CUDA 사용 가능:', torch.cuda.is_available())

## 2. 코드 받기 (GitHub clone)

In [ ]:
!git clone https://github.com/ins420/scrfd-proface.git
%cd scrfd-proface
!ls

## 3. 패키지 설치
insightface, onnxruntime-gpu 설치 (1~2분 소요). torch는 코랩 기본 GPU 버전 사용.

In [ ]:
!pip install -q insightface opencv-python-headless pycryptodome
# insightface가 CPU용 onnxruntime을 끌고 오므로 제거 후 GPU판만 설치
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q onnxruntime-gpu
print('설치 완료')

# GPU provider 확인
import onnxruntime as ort
print('available providers:', ort.get_available_providers())

## 4-1. demo.mp4 업로드
파일 선택 창이 뜨면 **영상 파일(mp4)** 만 선택하세요.

In [ ]:
from google.colab import files
import os

print('▶ demo.mp4 를 선택하세요')
up = files.upload()
for fn in up:
    print(f'  업로드됨: {fn}  ({os.path.getsize(fn)//1024} KB)')

## 4-2. INN 체크포인트(.pth) 업로드
파일 선택 창이 뜨면 **.pth 파일** 만 선택하세요. 자동으로 `checkpoints/` 로 이동됩니다.

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('checkpoints', exist_ok=True)
print('▶ .pth 파일을 선택하세요')
up = files.upload()
for fn in up:
    if fn.endswith('.pth'):
        shutil.move(fn, f'checkpoints/{fn}')
        print(f'  {fn}  →  checkpoints/')
    else:
        print(f'  무시됨(.pth 아님): {fn}')

# 체크포인트 경로 확인
import config as c
print('\nINN_CHECKPOINT =', c.INN_CHECKPOINT)
print('파일 존재:', os.path.exists(c.INN_CHECKPOINT))

In [ ]:
# 인자: demo.mp4 [프레임간격] [최대프레임수]
#  - 검증용으로 처음 600프레임(약 20초)만 처리 → 빠르게 결과 확인
#  - 전체 처리하려면 맨 끝 600 을 0 으로
!python batch_video.py demo.mp4 1 600

In [ ]:
!python batch_video.py demo.mp4 1

## 6. 결과 미리보기 (비교 영상)

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# 용량이 크면 미리보기가 느릴 수 있음 → 다운로드(7번)를 권장
mp4 = open('out_compare.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'<video width=900 controls><source src="{data_url}" type="video/mp4"></video>')

## 7. 결과 다운로드

In [ ]:
from google.colab import files
files.download('out_compare.mp4')     # 원본|익명화|복원 비교
files.download('out_anonymized.mp4')  # 익명화 영상
files.download('out_restored.mp4')    # 복원 영상